# Silver Join - Telemetry and Space Weather

Builds an hourly Silver table by enriching JPL ephemeris vectors with nearby DONKI space-weather events.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "orion" / "config.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break


In [ ]:
from src.orion.transforms.mission_timeline import (
    DEFAULT_JOIN_WINDOW_HOURS,
    donki_space_weather_events_table_name,
    jpl_ephemeris_vectors_table_name,
    rebuild_telemetry_space_weather_hourly,
    telemetry_space_weather_hourly_table_name,
)

source_ephemeris_table = jpl_ephemeris_vectors_table_name()
source_space_weather_table = donki_space_weather_events_table_name()
target_table = telemetry_space_weather_hourly_table_name()

print(f"Ephemeris source: {source_ephemeris_table}")
print(f"Space-weather source: {source_space_weather_table}")
print(f"Target table: {target_table}")
print(f"Join window hours: {DEFAULT_JOIN_WINDOW_HOURS}")


In [ ]:
result = rebuild_telemetry_space_weather_hourly(
    spark,
    source_ephemeris_table=source_ephemeris_table,
    source_space_weather_table=source_space_weather_table,
    target_table=target_table,
)

print(f"Rows written: {result['rows_written']}")


In [ ]:
display(
    spark.sql(f"""
        SELECT
            observation_timestamp_utc,
            target_body,
            nearby_space_weather_event_count,
            nearby_event_types_csv,
            nearby_flare_classes_csv,
            has_nearby_m_class_flare,
            has_nearby_x_class_flare
        FROM {target_table}
        ORDER BY observation_timestamp_utc
        LIMIT 50
    """)
)
